# Steganographic Detection

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.fft import fft2, dct
from scipy.stats import chi2, entropy, kurtosis, skew
import hashlib
from collections import defaultdict

class SteganographicAnalysisEngine:
    def __init__(self):
        self.lsb_detector = OneClassSVM(nu=0.1, kernel='rbf', gamma='scale')
        self.frequency_detector = IsolationForest(contamination=0.15, random_state=42)
        self.scaler = StandardScaler()
        self.stego_signatures = defaultdict(list)
        self.baseline_statistics = {}

    def extract_lsb_features(self, image_data):
        if len(image_data.shape) == 3:
            image_data = np.mean(image_data, axis=2)

        lsb_plane = image_data.astype(np.uint8) & 1
        lsb_histogram = np.bincount(lsb_plane.flatten(), minlength=2)
        lsb_entropy = entropy(lsb_histogram + 1e-10)

        pairs_analysis = self.pairs_of_values_analysis(image_data)
        sample_pairs = self.sample_pairs_analysis(image_data)

        autocorrelation = np.corrcoef(lsb_plane[:-1, :-1].flatten(), lsb_plane[1:, 1:].flatten())[0, 1]
        lsb_variance = np.var(lsb_plane)

        return [lsb_entropy, pairs_analysis, sample_pairs, autocorrelation, lsb_variance]

    def pairs_of_values_analysis(self, image):
        pixels = image.flatten().astype(int)
        even_pixels = pixels[pixels % 2 == 0]
        odd_pixels = pixels[pixels % 2 == 1]

        expected_ratio = len(even_pixels) / max(len(odd_pixels), 1)
        chi2_statistic = 0

        for i in range(0, min(len(pixels), 256), 2):
            observed_even = np.sum(pixels == i)
            observed_odd = np.sum(pixels == i + 1)
            expected = (observed_even + observed_odd) / 2

            if expected > 0:
                chi2_statistic += ((observed_even - expected) ** 2) / expected
                chi2_statistic += ((observed_odd - expected) ** 2) / expected

        return chi2_statistic / max(len(pixels), 1)

    def sample_pairs_analysis(self, image):
        pixels = image.flatten()
        differences = []

        for i in range(len(pixels) - 1):
            if pixels[i] % 2 == 0 and pixels[i+1] % 2 == 1:
                differences.append(abs(pixels[i+1] - pixels[i]))
            elif pixels[i] % 2 == 1 and pixels[i+1] % 2 == 0:
                differences.append(abs(pixels[i+1] - pixels[i]))

        return np.mean(differences) if differences else 0

    def frequency_domain_analysis(self, image_data):
        if len(image_data.shape) == 3:
            image_data = np.mean(image_data, axis=2)

        dct_coeffs = dct(dct(image_data.T, norm='ortho').T, norm='ortho')
        fft_coeffs = np.abs(fft2(image_data))

        dct_entropy = entropy(np.abs(dct_coeffs).flatten() + 1e-10)
        fft_entropy = entropy(fft_coeffs.flatten() + 1e-10)

        high_freq_energy = np.sum(fft_coeffs[fft_coeffs.shape[0]//2:, fft_coeffs.shape[1]//2:])
        total_energy = np.sum(fft_coeffs)
        high_freq_ratio = high_freq_energy / max(total_energy, 1e-10)

        dct_kurtosis = kurtosis(dct_coeffs.flatten())
        dct_skewness = skew(dct_coeffs.flatten())

        return [dct_entropy, fft_entropy, high_freq_ratio, dct_kurtosis, dct_skewness]

    def detect_f5_steganography(self, image_data):
        if len(image_data.shape) == 3:
            image_data = np.mean(image_data, axis=2)

        dct_blocks = []
        for i in range(0, image_data.shape[0] - 8, 8):
            for j in range(0, image_data.shape[1] - 8, 8):
                block = image_data[i:i+8, j:j+8]
                dct_block = dct(dct(block.T, norm='ortho').T, norm='ortho')
                dct_blocks.append(dct_block.flatten())

        if not dct_blocks:
            return 0

        dct_matrix = np.array(dct_blocks)
        zero_coeffs = np.sum(np.abs(dct_matrix) < 1e-6)
        total_coeffs = dct_matrix.size

        histogram_anomaly = self.detect_histogram_anomalies(dct_matrix)
        return (zero_coeffs / total_coeffs) + histogram_anomaly

    def detect_histogram_anomalies(self, dct_coeffs):
        flattened = dct_coeffs.flatten()
        hist, bins = np.histogram(flattened, bins=50)

        expected_hist = np.full_like(hist, np.mean(hist))
        chi2_stat = np.sum((hist - expected_hist) ** 2 / (expected_hist + 1e-10))

        return min(chi2_stat / len(hist), 1.0)

    def advanced_statistical_analysis(self, image_data):
        if len(image_data.shape) == 3:
            channels = [image_data[:,:,i] for i in range(image_data.shape[2])]
        else:
            channels = [image_data]

        channel_correlations = []
        pixel_variances = []
        gradient_magnitudes = []

        for channel in channels:
            if len(channels) > 1:
                other_channels = [c for c in channels if not np.array_equal(c, channel)]
                if other_channels:
                    correlation = np.corrcoef(channel.flatten(), other_channels[0].flatten())[0, 1]
                    channel_correlations.append(correlation)

            pixel_variances.append(np.var(channel))

            grad_x = np.diff(channel, axis=1)
            grad_y = np.diff(channel, axis=0)
            gradient_mag = np.mean(np.sqrt(grad_x[:, :-1]**2 + grad_y[:-1, :]**2))
            gradient_magnitudes.append(gradient_mag)

        return {
            'channel_correlations': channel_correlations,
            'pixel_variances': pixel_variances,
            'gradient_magnitudes': gradient_magnitudes
        }

    def comprehensive_stego_detection(self, image_data):
        lsb_features = self.extract_lsb_features(image_data)
        freq_features = self.frequency_domain_analysis(image_data)
        f5_score = self.detect_f5_steganography(image_data)
        statistical_features = self.advanced_statistical_analysis(image_data)

        combined_features = lsb_features + freq_features + [f5_score]
        combined_features.extend(statistical_features['pixel_variances'])
        combined_features.extend(statistical_features['gradient_magnitudes'])

        if len(combined_features) < 10:
            combined_features.extend([0] * (10 - len(combined_features)))

        features_array = np.array(combined_features[:15]).reshape(1, -1)
        features_scaled = self.scaler.fit_transform(features_array)

        lsb_anomaly = self.lsb_detector.fit(features_scaled).decision_function(features_scaled)[0]
        freq_anomaly = self.frequency_detector.fit(features_scaled).decision_function(features_scaled)[0]

        stego_probability = 1 / (1 + np.exp(-(lsb_anomaly + freq_anomaly + f5_score * 2)))

        detection_confidence = 'HIGH' if stego_probability > 0.7 else 'MEDIUM' if stego_probability > 0.4 else 'LOW'

        return {
            'steganography_detected': stego_probability > 0.5,
            'confidence': detection_confidence,
            'probability': stego_probability,
            'lsb_score': lsb_anomaly,
            'frequency_score': freq_anomaly,
            'f5_score': f5_score,
            'features': combined_features
        }